In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import talib.abstract as ta
import scipy as sp
from sklearn.preprocessing import PowerTransformer
from typing import List
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [2]:
# Apple Futures data using yfinance
ticker = 'GARAN.IS'
data = yf.download(ticker, start="2015-01-01", interval="1D")

# Check the data
display(data.head())

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,GARAN.IS,GARAN.IS,GARAN.IS,GARAN.IS,GARAN.IS
Date,,,,,
2015-01-01,6.962500,6.962500,6.962500,6.962500,0
2015-01-02,6.955109,6.999457,6.925544,6.955109,20815078
2015-01-05,7.058586,7.125106,7.021630,7.036412,44673228
2015-01-06,7.102932,7.110324,7.029021,7.051194,41805364
2015-01-07,7.125106,7.199018,7.058586,7.102932,54157823


In [3]:
data.columns = ['Close','High','Low','Open','Volume']

In [4]:
data_copy = data.copy()

# Preprocess
## Previous High & Low columns

In [5]:
data_copy['Previous High'] = data_copy['High'].shift(1)
data_copy['Previous Low'] = data_copy['Low'].shift(1) 

## Trend Type

In [6]:
def trend_type(df, 
                     atr_len=20, atr_ma_len=20, 
                     adx_len=14, di_len=14, 
                     adx_limit=25, smooth_factor=3, 
                     use_atr=True, use_adx=True): 
    # Convert data type to float  
    high = df['High'].values.astype(float)
    low = df['Low'].values.astype(float)
    close = df['Close'].values.astype(float)

    # ATR & ATR MA (SMA) 
    df['ATR'] = ta.ATR(high, low, close, timeperiod=atr_len)

    df['ATR_MA'] = ta.SMA(df['ATR'].values, timeperiod=atr_ma_len)
    
    # ADX & DI (+DI, -DI) 
    df['ADX'] = ta.ADX(high, low, close, timeperiod=adx_len)
    df['plusDI'] = ta.PLUS_DI(high, low, close, timeperiod=di_len)
    df['minusDI'] = ta.MINUS_DI(high, low, close, timeperiod=di_len)
    
    # Sideways
    cond_sideways_atr = (df['ATR'] <= df['ATR_MA']) if use_atr else False
    cond_sideways_adx = (df['ADX'] <= adx_limit) if use_adx else False
    
    df['is_sideways'] = cond_sideways_atr | cond_sideways_adx
    
    # Trend type change
    # plusDI > minusDI -> Uptrend
    df['is_up'] = df['plusDI'] > df['minusDI']
    
    # 0: Sideways, 2: Up, -2: Down
    df['trend_type'] = np.where(df['is_sideways'], 0, 
                                np.where(df['is_up'], 2, -2))
    
    df.loc[df['ADX'].isna() | df['ATR'].isna(), 'trend_type'] = np.nan
    
    # Smoothing
    # Orijinal script: sma(trendType, smooth) / 2 * 2
    sma_trend = ta.SMA(df['trend_type'].values, timeperiod=smooth_factor)
    df['smooth_type'] = np.round(sma_trend / 2) * 2
    
    return df

In [7]:
data_copy = trend_type(data_copy)

In [8]:
data_copy.head()

,Close,High,Low,Open,Volume,Previous High,Previous Low,ATR,ATR_MA,ADX,plusDI,minusDI,is_sideways,is_up,trend_type,smooth_type
Date,,,,,,,,,,,,,,,,
2015-01-01,6.962500,6.962500,6.962500,6.962500,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
2015-01-02,6.955109,6.999457,6.925544,6.955109,20815078,6.962500,6.962500,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
2015-01-05,7.058586,7.125106,7.021630,7.036412,44673228,6.999457,6.925544,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
2015-01-06,7.102932,7.110324,7.029021,7.051194,41805364,7.125106,7.021630,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN
2015-01-07,7.125106,7.199018,7.058586,7.102932,54157823,7.110324,7.029021,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN


In [9]:
def visualize_full_analysis(df, lag=8):
    dff = df.copy()
    
    # Lag Uygulaması: Görsel hizalama için verileri sola kaydırıyoruz
    # Not: Bu sadece görsel analiz içindir, canlı işlemde shift kullanılmaz.
    dff['lagged_smooth'] = dff['smooth_type'].shift(-lag)

    # 4 Panelli yapı
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.02, 
                        subplot_titles=('Price & Trend', 'Trend Typed(Lagged)', 
                                        'ADX & DI (+/-)', 'ATR & ATR MA'),
                        row_heights=[0.5, 0.15, 0.15, 0.15])

    # 1. PANEL: MUM GRAFİĞİ
    fig.add_trace(go.Candlestick(
        x=dff.index, open=dff['Open'], high=dff['High'], 
        low=dff['Low'], close=dff['Close'], name='Price'
    ), row=1, col=1)

    # --- Üst Grafiğe Lag Düzeltmeli Arka Plan Ekleme ---
    dff['trend_change'] = dff['lagged_smooth'].diff().fillna(0) != 0
    change_indices = dff.index[dff['trend_change']].tolist()
    start_idx = dff.index[0]
    
    for end_idx in change_indices + [dff.index[-1]]:
        current_val = dff.loc[start_idx, 'lagged_smooth']
        if current_val == 2: # Uptrend
            fig.add_vrect(x0=start_idx, x1=end_idx, fillcolor="green", opacity=0.12, line_width=0, row=1, col=1)
        elif current_val == -2: # Downtrend
            fig.add_vrect(x0=start_idx, x1=end_idx, fillcolor="red", opacity=0.12, line_width=0, row=1, col=1)
        start_idx = end_idx

    # 2. PANEL: TREND TİPİ OSİLATÖRÜ (Lagged)
    fig.add_trace(go.Scatter(
        x=dff.index, 
        y=dff['lagged_smooth'], 
        mode='lines',
        line=dict(width=3, shape='hv', color='white'),
        name=f'Trend (Lag {lag})'
    ), row=2, col=1)

    # 3. PANEL: ADX & DI
    fig.add_trace(go.Scatter(x=dff.index, y=dff['ADX'], line=dict(color='cyan', width=2), name='ADX'), row=3, col=1)
    fig.add_trace(go.Scatter(x=dff.index, y=dff['plusDI'], line=dict(color='green', width=1.5), name='+DI'), row=3, col=1)
    fig.add_trace(go.Scatter(x=dff.index, y=dff['minusDI'], line=dict(color='red', width=1.5), name='-DI'), row=3, col=1)
    fig.add_hline(y=25, line_dash="dash", line_color="gray", opacity=0.5, row=3, col=1)

    # 4. PANEL: ATR & ATR MA
    fig.add_trace(go.Scatter(x=dff.index, y=dff['ATR'], line=dict(color='orange', width=2), name='ATR'), row=4, col=1)
    fig.add_trace(go.Scatter(x=dff.index, y=dff['ATR_MA'], line=dict(color='yellow', width=1, dash='dot'), name='ATR MA'), row=4, col=1)

    # TASARIM AYARLARI
    fig.update_layout(
        height=1100,
        template='plotly_dark',
        xaxis_rangeslider_visible=False,
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    fig.show()

In [10]:
visualize_full_analysis(data_copy.iloc[-500:,:])

## Runaway Gap

In [11]:
def runaway_gap(data: pd.DataFrame)->pd.DataFrame:
    up_trend_condition = (
        data['smooth_type'] == 2
    )
    data['previous_close'] = data['Close'].shift(1)
    runaway_gap_condition = (
        data['Open'] > data['previous_close']
    )
    data['is_runaway'] = 0
    data.loc[up_trend_condition & runaway_gap_condition, 'is_runaway'] = 1
    return data

In [12]:
data_copy = runaway_gap(data_copy)

In [13]:
data_copy[data_copy['is_runaway'] == 1]

,Close,High,Low,Open,Volume,Previous High,Previous Low,ATR,ATR_MA,ADX,plusDI,minusDI,is_sideways,is_up,trend_type,smooth_type,previous_close,is_runaway
Date,,,,,,,,,,,,,,,,,,
2015-03-23,6.770329,6.799894,6.718591,6.785112,33266270,6.718591,6.637287,0.190183,0.181389,29.496339,31.048383,24.398645,False,True,2.0,2.0,6.681634,1
2016-04-06,6.096464,6.254131,6.081448,6.246623,89116397,6.254130,6.149018,0.130465,0.121438,26.576793,29.966705,16.236637,False,True,2.0,2.0,6.231606,1
2016-04-07,6.043908,6.171544,6.028892,6.141512,84189827,6.254131,6.081448,0.131075,0.121767,26.230083,27.686507,17.804551,False,True,2.0,2.0,6.096464,1
2016-04-08,6.156528,6.186559,6.006368,6.066432,77292079,6.171544,6.028892,0.133531,0.122342,25.667694,25.089536,17.306965,False,True,2.0,2.0,6.043908,1
2016-04-19,6.614513,6.629528,6.546941,6.584481,79986832,6.561957,6.321703,0.134510,0.127194,28.089273,35.552938,12.198826,False,True,2.0,2.0,6.561957,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,137.199997,141.300003,137.000000,141.000000,13704507,141.600006,139.300003,4.473818,4.417976,36.939821,30.671395,16.720111,False,True,2.0,2.0,140.000000,1
2025-07-29,139.000000,140.199997,137.300003,137.600006,14950323,141.300003,137.000000,4.400127,4.420327,36.404003,29.187620,15.911251,True,True,0.0,2.0,137.199997,1
2025-08-06,149.600006,151.800003,147.300003,150.699997,23020977,152.699997,149.600006,4.431282,4.413733,40.207133,37.137208,13.794571,False,True,2.0,2.0,150.000000,1


In [14]:
def visualize_uptrend_analysis(df, lag=8):
    dff = df.copy()
    
    # Lag Uygulaması (Görsel hizalama için)
    dff['lagged_smooth'] = dff['smooth_type'].shift(-lag)

    fig = go.Figure()

    # 1. MUM GRAFİĞİ
    fig.add_trace(go.Candlestick(
        x=dff.index, open=dff['Open'], high=dff['High'], 
        low=dff['Low'], close=dff['Close'], name='Price'
    ))

    # --- SADECE UPTREND ARKA PLAN ---
    dff['trend_change'] = dff['lagged_smooth'].diff().fillna(0) != 0
    change_indices = dff.index[dff['trend_change']].tolist()
    start_idx = dff.index[0]
    
    for end_idx in change_indices + [dff.index[-1]]:
        current_val = dff.loc[start_idx, 'lagged_smooth']
        if current_val == 2: 
            fig.add_vrect(
                x0=start_idx, x1=end_idx, 
                fillcolor="green", opacity=0.15, 
                line_width=0, layer="below"
            )
        start_idx = end_idx

    # --- RUNAWAY SİNYALLERİ (Hataya Karşı Güvenli Yöntem) ---
    runaway_data = dff[dff['is_runaway'] == 1]
    
    # Her bir runaway sinyali için görünmez bir scatter izi üzerinden dikey çizgi simülasyonu
    for date in runaway_data.index:
        fig.add_shape(
            type="line",
            x0=date, x1=date,
            y0=0, y1=1,
            xref="x", yref="paper", # y ekseni 0-1 arası (tüm grafik boyu)
            line=dict(color="yellow", width=1.5, dash="dash"),
        )
        # Etiketi ayrı bir text katmanı olarak ekleyelim (Opsiyonel)
        fig.add_annotation(
            x=date, y=1,
            yref="paper",
            text="Runaway",
            showarrow=False,
            font=dict(color="yellow", size=10),
            textangle=-90,
            xanchor="right"
        )

    # TASARIM AYARLARI
    fig.update_layout(
        title="Uptrend & Runaway Analysis",
        height=800,
        template='plotly_dark',
        xaxis_rangeslider_visible=False,
        yaxis_title="Price",
        showlegend=True
    )

    fig.show()

In [15]:
visualize_uptrend_analysis(data_copy.iloc[-50:,:])

In [18]:
import yfinance as yf
import pandas as pd
import numpy as np
import talib.abstract as ta

def analyze_stock(ticker, start_date, end_date):
    # 1. Verileri Çekme
    print(f"{ticker} verileri yfinance üzerinden çekiliyor...")
    df = yf.download(ticker, start=start_date, end=end_date)
    
    # 2. Teknik Göstergeler (SMA ve RSI)
    # Hareketli ortalamalar için pandas rolling kullanılmaya devam ediliyor, 
    # ancak istersen bunları da ta.SMA() olarak değiştirebilirsin.
    df['SMA_20'] = df['Close'].rolling(window=20).mean()
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    
    # TA-Lib (abstract) kullanarak RSI hesaplama
    # yfinance sütunları büyük harfle (Close) getirdiği için direkt seriyi veriyoruz
    df['RSI'] = ta.RSI(df['Close'], timeperiod=14)
    
    df['Vol_SMA_20'] = df['Volume'].rolling(window=20).mean()

    # 3. Trend Analizi (Up, Down, Sideway)
    # Basit yaklaşım: 20 günlük SMA > 50 günlük SMA ise Uptrend, tersi Downtrend, 
    # Fiyat hareketleri dar bir banttaysa (SMA'lar birbirine çok yakınsa) Sideway.
    df['Trend'] = 'Sideway'
    
    # SMA'lar arası fark %2'den büyükse net bir trend vardır varsayımı
    trend_threshold = 0.02 
    sma_diff = (df['SMA_20'] - df['SMA_50']) / df['SMA_50']
    
    df.loc[sma_diff > trend_threshold, 'Trend'] = 'Up'
    df.loc[sma_diff < -trend_threshold, 'Trend'] = 'Down'

    # 4. Geri Çekilme (Pullback) Noktaları
    # Bulunduğu trendden sayılır: Ana trend 'Up' iken fiyat geçici olarak 20 SMA'nın altına düşerse bu bir pullback'tir.
    df['Pullback'] = False
    df.loc[(df['Trend'] == 'Up') & (df['Low'] < df['SMA_20']) & (df['Close'] > df['SMA_50']), 'Pullback'] = True
    df.loc[(df['Trend'] == 'Down') & (df['High'] > df['SMA_20']) & (df['Close'] < df['SMA_50']), 'Pullback'] = True

    # 5. Pivot Noktaları, Destek (Support) ve Direnç (Resistance)
    # Pivot sadece önceki günün verisiyle hesaplanır.
    df['Pivot'] = (df['High'].shift(1) + df['Low'].shift(1) + df['Close'].shift(1)) / 3
    df['R1'] = (2 * df['Pivot']) - df['Low'].shift(1)
    df['S1'] = (2 * df['Pivot']) - df['High'].shift(1)
    
    # İstenen kural: Sadece Sideway trendlerde destek ve direnci dikkate al
    df['Sideway_Support'] = np.where(df['Trend'] == 'Sideway', df['S1'], np.nan)
    df['Sideway_Resistance'] = np.where(df['Trend'] == 'Sideway', df['R1'], np.nan)

    # 6. Gap (Boşluk) Analizi ve Sınıflandırma
    df['Gap_Type'] = 'None'
    
    # Gap şartları (Yukarı veya Aşağı)
    gap_up = df['Low'] > df['High'].shift(1)
    gap_down = df['High'] < df['Low'].shift(1)
    is_gap = gap_up | gap_down

    prev_trend = df['Trend'].shift(1)
    
    for i in range(1, len(df)):
        if is_gap.iloc[i]:
            current_trend_val = df['Trend'].iloc[i]
            prev_trend_val = prev_trend.iloc[i]
            rsi_val = df['RSI'].iloc[i]
            vol_spike = df['Volume'].iloc[i] > (1.5 * df['Vol_SMA_20'].iloc[i]) # Hacim normalin 1.5 katıysa
            
            # 1. Common Gap (Sıradan Boşluk): Yatay piyasada düşük/normal hacimle olur.
            if prev_trend_val == 'Sideway' and current_trend_val == 'Sideway':
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Common Gap'
            
            # 2. Breakaway Gap (Kopuş Boşluğu): Yatay piyasadan çıkışta, yüksek hacimle olur.
            elif prev_trend_val == 'Sideway' and current_trend_val != 'Sideway' and vol_spike:
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Breakaway Gap'
            
            # 3. Exhaustion Gap (Tükeniş Boşluğu): Trendin sonlarında, aşırı alım/satım bölgelerinde yüksek hacimle olur.
            elif (prev_trend_val == 'Up' and rsi_val > 70 and vol_spike) or \
                 (prev_trend_val == 'Down' and rsi_val < 30 and vol_spike):
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Exhaustion Gap'
            
            # 4. Runaway Gap (Kaçış/Ölçüm Boşluğu): Güçlü bir trendin ortasında meydana gelir.
            elif (prev_trend_val == 'Up' and current_trend_val == 'Up') or \
                 (prev_trend_val == 'Down' and current_trend_val == 'Down'):
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Runaway Gap'
            
            else:
                df.iloc[i, df.columns.get_loc('Gap_Type')] = 'Unclassified Gap'

    return df

# Örnek Kullanım: Apple hissesini son 1 yıl için analiz edelim
if __name__ == "__main__":
    pd.set_option('display.max_columns', None)
    
    # Hisseden veriyi çek ve analiz et
    result_df = analyze_stock("AAPL", "2023-01-01", "2024-01-01")
    
    # Sadece aksiyon olan günleri (Pullback veya Gap) filtreleyip ekrana yazdıralım
    action_days = result_df[(result_df['Pullback'] == True) | (result_df['Gap_Type'] != 'None')].copy()
    
    print("\n--- Önemli Hareketlerin Olduğu Günler (Pullback & Gaps) ---")
    columns_to_show = ['Close', 'Trend', 'Pullback', 'Sideway_Support', 'Sideway_Resistance', 'Gap_Type']
    print(action_days[columns_to_show].tail(20)) # Son 20 aksiyon gününü göster

[*********************100%***********************]  1 of 1 completed

AAPL verileri yfinance üzerinden çekiliyor...


Exception: input_arrays parameter missing required data key: close